In [3]:
from datasets import load_dataset, DatasetDict
from local_rag.core.text_cleaning import clean_text
from local_rag.utils.logger import get_logger
import time
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path


cwd = Path.cwd()
load_dotenv(cwd / "../../.env")
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)

logger = get_logger(__name__)
DATASET_NAME = "PrimeQA/clapnq"
USER_NAME = "joshuale"
CLEANED_DATASET_NAME = f"{USER_NAME}/clapnq_cleaned"
SAMPLE_SIZE = 100

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
def clean_batch_benchmark(batch):
    # Clean the 'text' field (if it exists)
    if 'text' in batch:
        texts = batch["text"]
        cleaned_texts = []
        for text in texts:
            cleaned_text = clean_text(text)
            cleaned_texts.append(cleaned_text)
        batch["text"] = cleaned_texts
    
    # Clean the 'output' field with nested structure
    if 'output' in batch:
        outputs = batch['output']
        cleaned_outputs = []
        
        for output in outputs:
            cleaned_output = output.copy()  # Keep original structure
            
            # Clean the 'answer' field
            if 'answer' in output and output['answer']:
                cleaned_output['answer'] = clean_text(output['answer'])
            
            # Clean the 'selected_sentences' field (list of strings)
            if 'selected_sentences' in output and output['selected_sentences']:
                cleaned_sentences = []
                for sentence in output['selected_sentences']:
                    if sentence:  # Check if sentence is not empty
                        cleaned_sentences.append(clean_text(sentence))
                    else:
                        cleaned_sentences.append(sentence)
                cleaned_output['selected_sentences'] = cleaned_sentences
            
            cleaned_outputs.append(cleaned_output)
        
        batch['output'] = cleaned_outputs
    
    return batch

## 1. Loading Corpus

In [3]:
# dataset = load_dataset(DATASET_NAME, split="train")
dataset_dict = load_dataset(DATASET_NAME)

logger.info(f"Available splits: {list(dataset_dict.keys())}")
logger.info(f"Train size: {len(dataset_dict['train'])}")
logger.info(f"Validation size: {len(dataset_dict['validation'])}")

2026-01-09 00:13:53 INFO     Available splits: ['train', 'validation']

                    INFO     Train size: 3745

                    INFO     Validation size: 600

## 2. Clean a Sample

In [7]:
# Test cleaning on samples from both splits
train_sample = dataset_dict['train'].select(indices=range(min(SAMPLE_SIZE, len(dataset_dict['train']))))
val_sample = dataset_dict['validation'].select(indices=range(min(SAMPLE_SIZE, len(dataset_dict['validation']))))

In [13]:
# Test cleaning on a small sample first
logger.info("Testing cleaning on small samples...")

# Clean train sample
start = time.time()
cleaned_train_sample = train_sample.map(
    clean_batch_benchmark,
    batched=True,
    batch_size=50,
    num_proc=2,
)
train_sample_time = time.time() - start
logger.info(f"Cleaning {len(train_sample)} train samples took {train_sample_time:.2f} seconds")

# Clean validation sample  
start = time.time()
cleaned_val_sample = val_sample.map(
    clean_batch_benchmark,
    batched=True,
    batch_size=50,
    num_proc=2,
)
val_sample_time = time.time() - start
logger.info(f"Cleaning {len(val_sample)} validation samples took {val_sample_time:.2f} seconds")

2026-01-09 00:21:22 INFO     Testing cleaning on small samples...

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

                    INFO     Cleaning 100 train samples took 0.17 seconds

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

2026-01-09 00:21:23 INFO     Cleaning 100 validation samples took 0.16 seconds

## 3. Clean Both Splits

In [ ]:
# Clean the entire dataset (both splits)
logger.info("Starting to clean the entire dataset...")

# Clean train split
start = time.time()
cleaned_train = dataset_dict['train'].map(
    clean_batch_benchmark,
    batched=True,
    batch_size=1000,
    num_proc=4,
)
train_time = time.time() - start
logger.info(f"Cleaning {len(dataset_dict['train'])} train samples took {train_time:.2f} seconds")

# Clean validation split
start = time.time()
cleaned_val = dataset_dict['validation'].map(
    clean_batch_benchmark,
    batched=True,
    batch_size=1000,
    num_proc=4,
)
val_time = time.time() - start
logger.info(f"Cleaning {len(dataset_dict['validation'])} validation samples took {val_time:.2f} seconds")

total_time = train_time + val_time
logger.info(f"Total cleaning time: {total_time:.2f} seconds")

2026-01-09 00:21:58 INFO     Starting to clean the entire dataset...

Map (num_proc=4):   0%|          | 0/3745 [00:00<?, ? examples/s]

2026-01-09 00:21:59 INFO     Cleaning 3745 train samples took 0.23 seconds

Map (num_proc=4):   0%|          | 0/600 [00:00<?, ? examples/s]

                    INFO     Cleaning 600 validation samples took 0.19 seconds

                    INFO     Total cleaning time: 0.42 seconds

## 4. Push to HF

In [15]:
# Create cleaned dataset dict with both splits
cleaned_dataset_dict = DatasetDict({
    'train': cleaned_train,
    'validation': cleaned_val
})

# Push to hub with both splits
logger.info(f"Pushing cleaned dataset to {CLEANED_DATASET_NAME}...")
cleaned_dataset_dict.push_to_hub(
    CLEANED_DATASET_NAME,
    private=True,
    max_shard_size="5GB",
)
logger.info("Dataset successfully pushed to Hugging Face Hub!")

# Print final dataset info
print(f"Cleaned dataset uploaded to: {CLEANED_DATASET_NAME}")
print(f"Train split: {len(cleaned_train)} samples")
print(f"Validation split: {len(cleaned_val)} samples")

2026-01-09 00:22:35 INFO     Pushing cleaned dataset to joshuale/clapnq_cleaned...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

2026-01-09 00:22:50 INFO     Dataset successfully pushed to Hugging Face Hub!

Cleaned dataset uploaded to: joshuale/clapnq_cleaned
Train split: 3745 samples
Validation split: 600 samples
